# BERTopic Taxonomy — Swiss Legal Corpus

Goal: Discover data-driven topic clusters in `laws_de.csv` using BERTopic (BERT embeddings → UMAP → HDBSCAN).

Sections:
1. Setup & device detection (MPS / CUDA / CPU)
2. Data loading & law code parsing
3. Natural topic count prior from corpus structure
4. Embedding model survey (MiniLM vs gbert-base vs bge-m3) on a sample
5. Full BERTopic fit with HDBSCAN (auto topic count) + min_cluster_size sweep
6. Post-hoc topic reduction & coherence elbow
7. Topic analysis & comparison to hand-crafted taxonomy
8. Law code → topic mapping
9. Query → topic classification (val.csv)
10. Save artifacts

## Section 1 — Setup & Device Detection

In [ ]:
import os, time, warnings, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter

warnings.filterwarnings('ignore')

# --- device ---
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'device: {DEVICE}')
print(f'torch:  {torch.__version__}')

BASE     = '../data'
OUT_DIR  = '../data/bertopic'
os.makedirs(OUT_DIR, exist_ok=True)

## Section 2 — Data Loading & Law Code Parsing

In [ ]:
laws_df = pd.read_csv(f'{BASE}/laws_de.csv')
val_df  = pd.read_csv(f'{BASE}/val.csv')

laws_df['text']  = laws_df['text'].fillna('')
laws_df['title'] = laws_df['title'].fillna('')

# Extract numeric Swiss law code from citation end, e.g. 'Art. 5 Abs. 1 641.711' -> '641.711'
laws_df['law_code'] = laws_df['citation'].str.extract(r'(\d[\d\.]*\d|\d)\s*$')

print(f'laws:      {len(laws_df):,} articles, {laws_df["law_code"].nunique()} unique law codes')
print(f'val:       {len(val_df)} queries')
print(f'text len:  mean={laws_df["text"].str.len().mean():.0f}  median={laws_df["text"].str.len().median():.0f}  max={laws_df["text"].str.len().max()}')
laws_df.head(3)

## Section 3 — Natural Topic Count Prior from Corpus Structure

In [ ]:
# How many unique law codes, and how are articles distributed?
code_counts = laws_df['law_code'].value_counts()
print(f'Unique law codes: {code_counts.shape[0]}')
print(f'Articles per law code:  mean={code_counts.mean():.1f}  median={code_counts.median():.0f}  max={code_counts.max()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(code_counts.values, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Articles per law code')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of articles per law code')

axes[1].plot(sorted(code_counts.values, reverse=True))
axes[1].set_xlabel('Law code rank')
axes[1].set_ylabel('Article count')
axes[1].set_title('Sorted article counts (law-code Zipf)')
plt.tight_layout()
plt.show()

# Top 20 law codes by article count
print('\nTop 20 law codes by article count:')
top20 = code_counts.head(20).reset_index()
top20.columns = ['law_code', 'n_articles']
# join first title seen for each code
first_titles = laws_df.groupby('law_code')['title'].first().reset_index()
top20 = top20.merge(first_titles, on='law_code')
print(top20.to_string(index=False))

In [ ]:
# Swiss law code hierarchy: first segment of numeric code is top-level domain
# e.g. '1xx' = constitutional, '2xx' = private law, '3xx' = civil procedure, etc.
laws_df['domain'] = (
    laws_df['law_code']
    .str.extract(r'^(\d)', expand=False)   # Series, not DataFrame
    .fillna('?')
)

domain_labels = {
    '1': '1xx: State & Constitution',
    '2': '2xx: Private Law',
    '3': '3xx: Civil Procedure & Enforcement',
    '4': '4xx: Criminal Law',
    '5': '5xx: Administrative Law',
    '6': '6xx: Finance & Tax',
    '7': '7xx: Public Works & Environment',
    '8': '8xx: Health & Labor',
    '9': '9xx: Economic Law',
    '?': '?: Unknown / no code',
}

domain_counts = laws_df['domain'].value_counts().sort_index()
labels = [domain_labels.get(d, d) for d in domain_counts.index]

plt.figure(figsize=(10, 4))
plt.bar(labels, domain_counts.values, color='steelblue', edgecolor='white')
plt.xticks(rotation=20, ha='right', fontsize=8)
plt.ylabel('Article count')
plt.title('Articles by top-level Swiss law domain')
plt.tight_layout()
plt.show()

print('\nPrior: expect BERTopic to find ~20-80 coherent semantic topics (well below 984 law codes)')

## Section 4 — Embedding Model Survey on a Sample

In [ ]:
from sentence_transformers import SentenceTransformer

SAMPLE_N = 3000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(laws_df), size=SAMPLE_N, replace=False)
sample_df  = laws_df.iloc[sample_idx].reset_index(drop=True)
sample_texts = sample_df['text'].tolist()

def bench_embed(model_name, texts, device, batch_size=128):
    print(f'  loading {model_name} ...')
    m = SentenceTransformer(model_name, device=device)
    t0 = time.time()
    embs = m.encode(texts, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)
    elapsed = time.time() - t0
    dim = embs.shape[1]
    print(f'  -> {elapsed:.1f}s  dim={dim}  shape={embs.shape}')
    del m; gc.collect()
    return embs, elapsed

models_to_survey = [
    ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'MiniLM-multilingual'),
    # Uncomment to compare (slower):
    # ('deepset/gbert-base',  'gbert-base'),
    # ('BAAI/bge-m3',         'bge-m3'),
]

survey_results = {}
for model_name, label in models_to_survey:
    print(f'\n--- {label} ---')
    embs, elapsed = bench_embed(model_name, sample_texts, DEVICE)
    survey_results[label] = {'embs': embs, 'elapsed': elapsed, 'model_name': model_name}

print('\nSurvey done.')

In [ ]:
import umap

# UMAP 2D visualization of the sample, colored by top-level domain
sample_domains = sample_df['domain'].fillna('?').astype(str).values
unique_domains = sorted(set(sample_domains))
domain_colors  = {d: cm.tab10(i / len(unique_domains)) for i, d in enumerate(unique_domains)}

for label, res in survey_results.items():
    print(f'UMAP for {label} ...')
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    embs_2d = reducer.fit_transform(res['embs'])

    plt.figure(figsize=(10, 8))
    for d in unique_domains:
        mask = sample_domains == d
        plt.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
                    c=[domain_colors[d]], label=domain_labels.get(d, d),
                    s=5, alpha=0.6)
    plt.legend(fontsize=7, markerscale=3, loc='upper right')
    plt.title(f'UMAP of {SAMPLE_N} articles — {label}\n(colored by top-level law domain)')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/umap_sample_{label.replace("/","_")}.png', dpi=120)
    plt.show()
    print('saved.')

## Section 5 — Full BERTopic Fit (HDBSCAN + min_cluster_size sweep)

In [ ]:
# Embed full corpus with the chosen model (MiniLM-multilingual)
EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f'Embedding {len(laws_df):,} articles on {DEVICE} ...')
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

t0 = time.time()
corpus_embs = embedder.encode(
    laws_df['text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f'Done in {time.time()-t0:.1f}s  shape={corpus_embs.shape}')

# Save embeddings so we don't have to re-run
np.save(f'{OUT_DIR}/corpus_embs.npy', corpus_embs)
print('Saved corpus_embs.npy')

In [ ]:
# Run this cell instead of re-embedding if embeddings already exist
import os
embs_path = f'{OUT_DIR}/corpus_embs.npy'
if os.path.exists(embs_path):
    corpus_embs = np.load(embs_path)
    print(f'Loaded cached embeddings: {corpus_embs.shape}')
else:
    print('No cache found — run the embedding cell above first.')

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Configuration
MIN_CLUSTER_SIZE = 150   # main run; sweep done below
N_NEIGHBORS      = 15
N_COMPONENTS     = 10    # UMAP target dim for clustering

umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

# German general stop words + Swiss legal boilerplate
# Legal boilerplate appears in almost every article but carries no topic signal
STOP_WORDS = [
    # German function words
    'der', 'die', 'das', 'den', 'dem', 'des',
    'ein', 'eine', 'einer', 'einem', 'einen', 'eines',
    'und', 'oder', 'aber', 'nicht', 'auch', 'noch', 'nur', 'so',
    'ist', 'sind', 'wird', 'werden', 'hat', 'haben', 'wurde', 'wurden',
    'kann', 'können', 'muss', 'müssen', 'soll', 'sollen', 'darf', 'dürfen',
    'sich', 'aus', 'an', 'in', 'zu', 'von', 'mit', 'bei', 'nach',
    'durch', 'für', 'über', 'unter', 'vor', 'auf', 'ab', 'bis', 'um',
    'als', 'wie', 'wenn', 'dass', 'ob', 'im', 'am', 'beim', 'vom',
    'zur', 'zum', 'gegen', 'ohne', 'seit', 'es', 'er', 'sie', 'wir',
    'ihr', 'ich', 'du', 'ihn', 'ihm', 'ihnen', 'dieser', 'diese',
    'diesem', 'diesen', 'dieses', 'jede', 'jeder', 'jeden', 'jedem',
    'sowie', 'soweit', 'sofern', 'jedoch', 'dabei', 'davon', 'dazu',
    'darin', 'darüber', 'darunter', 'hierfür', 'hierbei', 'hierin',
    # Swiss legal structural boilerplate (appear in nearly every article)
    'artikel', 'art', 'absatz', 'abs', 'satz', 'buchstabe', 'ziffer',
    'ziff', 'paragraph', 'kapitel', 'titel', 'abschnitt', 'alinea',
    'littera', 'lit',
    # Legal filler verbs / phrases
    'gilt', 'gelten', 'geltenden', 'gemäss', 'betreffend', 'vorliegend',
    'anwendbar', 'vorgesehen', 'bestimmt', 'bezeichnet', 'geregelt',
]

vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.5,
    ngram_range=(1, 2),
    stop_words=STOP_WORDS,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    verbose=True,
    calculate_probabilities=False,
)

texts = laws_df['text'].tolist()
print(f'Fitting BERTopic on {len(texts):,} articles  (min_cluster_size={MIN_CLUSTER_SIZE}) ...')
t0 = time.time()
topics, _ = topic_model.fit_transform(texts, embeddings=corpus_embs)
print(f'Done in {time.time()-t0:.1f}s')

laws_df['topic'] = topics
n_topics   = topic_model.get_topic_info().shape[0] - 1  # exclude -1 (outlier)
n_outliers = (np.array(topics) == -1).sum()
print(f'Topics found:  {n_topics}')
print(f'Outlier docs:  {n_outliers:,} ({100*n_outliers/len(texts):.1f}%)')

In [ ]:
# min_cluster_size sweep to understand sensitivity
sweep_sizes   = [50, 100, 150, 200, 300, 500]
sweep_results = []

for mcs in sweep_sizes:
    hdb = HDBSCAN(min_cluster_size=mcs, metric='euclidean',
                  cluster_selection_method='eom', prediction_data=False)
    um  = UMAP(n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
               min_dist=0.0, metric='cosine', random_state=42)
    bm  = BERTopic(umap_model=um, hdbscan_model=hdb,
                   vectorizer_model=CountVectorizer(min_df=5, ngram_range=(1,2), stop_words=STOP_WORDS),
                   verbose=False, calculate_probabilities=False)
    t, _ = bm.fit_transform(texts, embeddings=corpus_embs)
    t_arr   = np.array(t)
    n_t     = bm.get_topic_info().shape[0] - 1
    out_pct = 100 * (t_arr == -1).sum() / len(t_arr)
    sweep_results.append({'mcs': mcs, 'n_topics': n_t, 'outlier_pct': out_pct})
    print(f'  mcs={mcs:4d}  topics={n_t:3d}  outliers={out_pct:.1f}%')
    del bm, um, hdb
    gc.collect()

sweep_df = pd.DataFrame(sweep_results)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(sweep_df['mcs'], sweep_df['n_topics'], 'o-b', label='# topics')
ax2.plot(sweep_df['mcs'], sweep_df['outlier_pct'], 's--r', label='outlier %')
ax1.set_xlabel('min_cluster_size')
ax1.set_ylabel('# topics', color='b')
ax2.set_ylabel('outlier %', color='r')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('min_cluster_size sweep')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/mcs_sweep.png', dpi=120)
plt.show()

## Section 6 — Post-hoc Topic Reduction & Coherence Elbow

In [ ]:
# Hierarchical topic reduction — find elbow in within-topic cosine similarity
topic_info_raw = topic_model.get_topic_info()
print(f'Before reduction: {len(topic_info_raw)-1} topics')
topic_info_raw.head(10)

In [ ]:
# BERTopic hierarchical merge — try a range of target topic counts
raw_n = len(topic_info_raw) - 1  # excluding outlier row
target_counts = list(range(max(5, raw_n // 4), raw_n, max(1, raw_n // 8)))

# Compute topic coherence proxy: mean pairwise cosine sim of top-10 word vectors
# (Simple: use BERTopic's built-in hierarchy distances)
hierarchical_topics = topic_model.hierarchical_topics(texts)
print('Hierarchical topics computed.')
print(hierarchical_topics.head(10))

In [ ]:
# Reduce to a reasonable number — elbow from hierarchy
# Plot the distance at which topics are merged to find natural elbow
distances = hierarchical_topics['Distance'].sort_values().values
plt.figure(figsize=(8, 4))
plt.plot(range(len(distances)), distances, marker='o', ms=3)
plt.xlabel('Merge step (ascending distance)')
plt.ylabel('Merge distance')
plt.title('Hierarchical merge distances — find elbow for target topic count')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/hierarchy_distances.png', dpi=120)
plt.show()

# Pick target: use elbow heuristic (largest gap in merge distances)
gaps = np.diff(distances)
elbow_step = np.argmax(gaps)  # step where merge distance jumps most
# Number of topics remaining after 'elbow_step' merges
target_n = raw_n - elbow_step
print(f'Elbow at merge step {elbow_step} → target_n = {target_n}')

In [ ]:
# Apply reduction
TARGET_N = max(10, min(target_n, 60))  # clamp to sensible range
print(f'Reducing to {TARGET_N} topics ...')
topic_model.reduce_topics(texts, nr_topics=TARGET_N)
reduced_topics = topic_model.topics_
laws_df['topic_reduced'] = reduced_topics

ti = topic_model.get_topic_info()
print(f'After reduction: {len(ti)-1} topics (excluding outlier)')
ti

## Section 7 — Topic Analysis & Comparison to Hand-crafted Taxonomy

In [ ]:
# Show top words per topic
print('Top 10 words per topic:\n')
for t_id in sorted(topic_model.get_topics().keys()):
    if t_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(t_id)[:10]]
    print(f'  Topic {t_id:3d}: {" | ".join(words)}')

In [ ]:
# Cross-tab: BERTopic topic vs. top-level law domain
crosstab = pd.crosstab(
    laws_df['topic_reduced'],
    laws_df['domain'],
    values=laws_df['citation'],
    aggfunc='count',
).fillna(0).astype(int)

# Rename columns with domain labels
crosstab.columns = [domain_labels.get(c, c) for c in crosstab.columns]

# Plot heatmap (normalized per topic row)
norm = crosstab.div(crosstab.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, max(6, len(crosstab) * 0.35)))
im = ax.imshow(norm.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(norm.columns)))
ax.set_xticklabels(norm.columns, rotation=30, ha='right', fontsize=7)
ax.set_yticks(range(len(norm.index)))
ax.set_yticklabels([f'T{i}' if i != -1 else 'outlier' for i in norm.index], fontsize=7)
ax.set_ylabel('BERTopic topic')
ax.set_xlabel('Swiss law domain')
ax.set_title('Topic vs domain overlap (row-normalized)')
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/topic_domain_heatmap.png', dpi=120)
plt.show()

In [ ]:
# Topic sizes
size_map = laws_df[laws_df['topic_reduced'] != -1]['topic_reduced'].value_counts().sort_index()

plt.figure(figsize=(max(8, len(size_map) * 0.5), 4))
plt.bar(size_map.index.astype(str), size_map.values, color='steelblue', edgecolor='white')
plt.xlabel('Topic ID')
plt.ylabel('Article count')
plt.title('Articles per topic (after reduction)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/topic_sizes.png', dpi=120)
plt.show()

print(f'Outlier articles: {(laws_df["topic_reduced"]==-1).sum():,}')

## Section 8 — Law Code → Topic Mapping

In [ ]:
# For each law code, find the plurality topic (ignoring outliers)
def plurality_topic(series):
    s = series[series != -1]
    if len(s) == 0:
        return -1
    return s.value_counts().idxmax()

lawcode_topic = (
    laws_df.groupby('law_code')['topic_reduced']
    .apply(plurality_topic)
    .rename('plurality_topic')
    .reset_index()
)

# Join first title for readability
lawcode_topic = lawcode_topic.merge(
    laws_df.groupby('law_code')['title'].first().reset_index(),
    on='law_code'
)

# Topic word labels
def topic_label(t_id):
    if t_id == -1:
        return 'outlier'
    words = [w for w, _ in topic_model.get_topic(t_id)[:5]]
    return f'T{t_id}: {" | ".join(words)}'

lawcode_topic['topic_label'] = lawcode_topic['plurality_topic'].apply(topic_label)

print(f'Law code → topic mapping: {len(lawcode_topic)} rows')
lawcode_topic.head(20)

In [ ]:
lawcode_topic.to_csv(f'{OUT_DIR}/lawcode_topic_mapping.csv', index=False)
print('Saved lawcode_topic_mapping.csv')

# How many law codes map to each topic?
print('\nLaw codes per topic:')
print(lawcode_topic['plurality_topic'].value_counts().sort_index())

## Section 9 — Query → Topic Classification (val.csv)

In [ ]:
# Embed val queries and assign them to the nearest topic
query_texts = val_df['query'].tolist()

print(f'Embedding {len(query_texts)} queries ...')
query_embs = embedder.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

# Use BERTopic's transform to assign topics
query_topics, _ = topic_model.transform(query_texts, embeddings=query_embs)
val_df['topic'] = query_topics
val_df['topic_label'] = [topic_label(t) for t in query_topics]

print('\nQuery topic assignments:')
val_df[['query_id', 'topic', 'topic_label']].to_string(index=False)

In [ ]:
# Check: do gold citations' law codes agree with query topic?
# For each query, get the set of topics for gold law codes
gold_topic_agreement = []
for _, row in val_df.iterrows():
    if pd.isna(row.get('gold_citations', '')):
        continue
    gold_codes = set()
    for cit in str(row['gold_citations']).split(';'):
        # extract numeric law code from gold citation
        m = __import__('re').search(r'(\d[\d\.]*\d|\d)\s*$', cit.strip())
        if m:
            gold_codes.add(m.group(1))
    # find topics of these law codes
    gold_topics = set()
    for code in gold_codes:
        match = lawcode_topic[lawcode_topic['law_code'] == code]
        if not match.empty:
            gold_topics.add(match.iloc[0]['plurality_topic'])
    query_t = row['topic']
    hit = query_t in gold_topics
    gold_topic_agreement.append({
        'query_id': row['query_id'],
        'query_topic': query_t,
        'gold_topics': sorted(gold_topics),
        'topic_hit': hit,
    })

agree_df = pd.DataFrame(gold_topic_agreement)
hit_rate = agree_df['topic_hit'].mean()
print(f'Query-topic → gold-citation-topic agreement: {hit_rate:.0%} ({agree_df["topic_hit"].sum()}/{len(agree_df)})')
print(agree_df.to_string(index=False))

## Section 10 — Save Artifacts

In [ ]:
# Save the BERTopic model
model_path = f'{OUT_DIR}/bertopic_model'
topic_model.save(model_path, serialization='safetensors', save_ctfidf=True,
                 save_embedding_model=EMBED_MODEL)
print(f'Model saved to {model_path}')

# Save laws_df with topic annotations
laws_df[['citation', 'law_code', 'domain', 'topic', 'topic_reduced']].to_csv(
    f'{OUT_DIR}/laws_with_topics.csv', index=False)
print('Saved laws_with_topics.csv')

# Save query topic assignments
val_df[['query_id', 'topic', 'topic_label']].to_csv(
    f'{OUT_DIR}/val_topic_assignments.csv', index=False)
print('Saved val_topic_assignments.csv')

# Summary
print('\n--- Summary ---')
print(f'Corpus articles:    {len(laws_df):,}')
print(f'Law codes:          {laws_df["law_code"].nunique()}')
print(f'BERTopic topics:    {len(topic_model.get_topics())-1}')
print(f'Outlier articles:   {(laws_df["topic_reduced"]==-1).sum():,}')
print(f'Output dir:         {OUT_DIR}')